In [122]:
import os
import json
import pandas as pd
folder_day = 'friday-enhance-portscan'
# 1. 设置包含多个 alert_json.txt 文件的文件夹路径
folder_path = f'../../alert_log_data/{folder_day}'

# 2. 获取所有以 alert_json.txt 开头的文件
file_list = [f for f in os.listdir(folder_path) if f.startswith('alert_json.txt')]

# 3. 初始化空的 DataFrame 用于合并数据
all_data = []

# 4. 遍历每个文件，读取 JSON 行并追加到列表中
for filename in file_list:
    file_path = os.path.join(folder_path, filename)
    with open(file_path, 'r') as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                all_data.append(data)
            except json.JSONDecodeError:
                print(f"跳过无效的 JSON 行：{line}")

# 5. 转换为 DataFrame
snort_df  = pd.DataFrame(all_data)

In [123]:
# 6. 读入真值标签数据
thurth_day = 'Friday'
truth_df = pd.read_csv(f'../../label_data/{thurth_day}-Attacks.pcap_ISCX.csv')

In [124]:
from datetime import timedelta

# 过滤截断数据包产生的误报告警，它们由snort本身配置错误产生
snort_df_filtered = snort_df[~((snort_df['proto'] == 'eth') & (snort_df['msg'].str.contains('IPv4 datagram length > captured length', na=False)))]
snort_df_filtered = snort_df_filtered.loc[snort_df_filtered['proto'] !='ARP']
snort_df_filtered.reset_index(inplace=True)

# 处理时间
snort_df_filtered.loc[:,'real_time'] = pd.to_datetime(snort_df_filtered['seconds'], unit='s',utc=True)
# snort日志数据，将日期时间转换为北美时区（例如 America/Toronto）
north_america_tz = 'America/Toronto'
snort_df_filtered['real_time'] = snort_df_filtered['real_time'].dt.tz_convert(north_america_tz)
# 手动加1小时修正偏差
snort_df_filtered['real_time'] = snort_df_filtered['real_time'] + timedelta(hours=1)
snort_df_filtered['real_time'] = snort_df_filtered['real_time'].dt.tz_localize(None)


# 真值标签数据，去掉列名两端空格
truth_df.columns = truth_df.columns.str.strip()
date_str = truth_df['Timestamp'].str.extract(r'(\d+/\d+/\d+)')[0]
time_str = truth_df['Timestamp'].str.extract(r'(\d+):(\d+)')
hours = time_str[0].astype(int)
minutes = time_str[1].astype(int)

# 假设从第1000行开始是下午，行号从0开始计数
afternoon_start_row_dict = {
    "Wednesday": 252656,
    "Friday": 1966,
}
afternoon_start_row = afternoon_start_row_dict[thurth_day]
# 复制小时数
hours_24 = hours.copy()
# 从指定行开始小时数+12
hours_24.iloc[afternoon_start_row:] = hours_24.iloc[afternoon_start_row:] + 12


# 组合成新的24小时格式时间
truth_df['Timestamp_dt'] = pd.to_datetime(
    date_str + ' ' + hours_24.astype(str) + ':' + minutes.astype(str).str.zfill(2),
    format='%d/%m/%Y %H:%M'
    )

In [127]:
# 3. 解析字段
truth_df['start_time'] = pd.to_datetime(truth_df['Timestamp_dt'])  # 12小时制已修复
truth_df['duration_s'] = truth_df['Flow Duration'] / 1000000  # 微秒 -> 秒

truth_df['end_time'] = truth_df['start_time'] + pd.to_timedelta(truth_df['duration_s'], unit='s')

proto_map = {0:'HOPOPT', 6: 'TCP', 17: 'UDP', 1: 'ICMP'}
truth_df['proto'] = truth_df['Protocol'].map(proto_map)

# 强制转换为整数类型，去掉小数点
snort_df_filtered['src_port'] = snort_df_filtered['src_port'].fillna(0).astype(int)
snort_df_filtered['dst_port'] = snort_df_filtered['dst_port'].fillna(0).astype(int)
snort_df_filtered = snort_df_filtered.drop(columns=['index'])

In [128]:
# 构造flow五元组匹配键，拼接字段作为字符串
def make_key(df):
    return df['Source IP'].astype(str) + '_' + df['Destination IP'].astype(str) + '_' + \
           df['Source Port'].astype(str) + '_' + df['Destination Port'].astype(str) + '_' + \
           df['proto'].astype(str)

truth_df['key'] = make_key(truth_df)
snort_df_filtered['key'] = make_key(snort_df_filtered.rename(columns={
    'src_addr': 'Source IP',
    'dst_addr': 'Destination IP',
    'src_port': 'Source Port',
    'dst_port': 'Destination Port',
    'proto': 'proto'
}))

In [129]:
from bisect import bisect_right

def find_label_for_row(row, truth_df):
    # 先找对应key的flows（小范围）
    flows = truth_df.get_group(row['key']) if row['key'] in truth_df.groups else None
    if flows is None:
        return 'BENIGN'
    #print('hello')
    # flows 按start_time排序，便于二分查找，允许有前后1分钟的误差
    starts = flows['start_time'].values - pd.Timedelta(minutes=1)
    ends = flows['end_time'].values + pd.Timedelta(minutes=1)
    labels = flows['Label'].values

    # 找出real_time对应的插入点
    pos = bisect_right(starts, row['real_time'])

    # 向前查找能匹配的区间
    for i in range(pos-1, -1, -1):
        #print('pos=',pos,'i=',i,'starts[i]=',starts[i],'row[real_time]=',row['real_time'],'ends[i]=',ends[i])
        if starts[i] <= row['real_time'] <= ends[i]:
            #print('Find labels! label = ',labels[i])
            return labels[i]
        if row['real_time'] > ends[i]:
            break
    return 'BENIGN'

In [130]:
# 5. 打标签
from tqdm import tqdm

tqdm.pandas(desc="Matching labels")
# 按key分组，生成分组对象
truth_df = truth_df.sort_values(['key', 'start_time'])
truth_grouped = truth_df.groupby('key')
snort_df_filtered['Label'] = snort_df_filtered.progress_apply(lambda row: find_label_for_row(row, truth_grouped), axis=1)

Matching labels: 100%|██████████| 180813/180813 [00:59<00:00, 3036.38it/s] 


In [131]:
snort_df_filtered['Label'].value_counts()

DDoS        112348
BENIGN       67470
Bot            566
PortScan       429
Name: Label, dtype: int64

In [133]:
print(f'周五，{folder_day}-alert-label标签结果：')
output_file_path = f'alert_label_merge_csv_correct/{folder_day}-alert-label.csv'
snort_df_correct = snort_df_filtered.copy()

from datetime import datetime
# 对误标注结果修正
# 周三DoS Hulk和Dos GoldenEye、DoS Slowhttptest修正
# start_time = datetime.strptime('2017-07-05 10:30:00', '%Y-%m-%d %H:%M:%S')
# end_time = datetime.strptime('2017-07-05 10:40:00', '%Y-%m-%d %H:%M:%S')
# conditional_Wednesday = (snort_df_filtered['Label'] == 'BENIGN') & (snort_df_filtered['real_time'] >= start_time) & (snort_df_filtered['real_time'] < end_time) & (snort_df_filtered['dst_addr']=='192.168.10.50') & (snort_df_filtered['dst_port']==80) & (snort_df_filtered['src_addr']=='172.16.0.1')
# snort_df_correct.loc[conditional_Wednesday, 'Label'] = 'DoS Slowhttptest'

# # 筛选10:40到11:10之间的数据
# start_time = datetime.strptime('2017-07-05 10:40:00', '%Y-%m-%d %H:%M:%S')
# end_time = datetime.strptime('2017-07-05 11:10:00', '%Y-%m-%d %H:%M:%S')
# conditional_Wednesday = (snort_df_filtered['Label'] == 'BENIGN') & (snort_df_filtered['real_time'] >= start_time) & (snort_df_filtered['real_time'] < end_time) & (snort_df_filtered['dst_addr']=='192.168.10.50') & (snort_df_filtered['dst_port']==80) & (snort_df_filtered['src_addr']=='172.16.0.1')
# snort_df_correct.loc[conditional_Wednesday, 'Label'] = 'DoS Hulk'

# start_time = datetime.strptime('2017-07-05 11:10:00', '%Y-%m-%d %H:%M:%S')
# end_time = datetime.strptime('2017-07-05 11:20:00', '%Y-%m-%d %H:%M:%S')
# conditional_Wednesday = (snort_df_filtered['Label'] == 'BENIGN') & (snort_df_filtered['real_time'] >= start_time) & (snort_df_filtered['real_time'] <= end_time) & (snort_df_filtered['dst_addr']=='192.168.10.50') & (snort_df_filtered['dst_port']==80) & (snort_df_filtered['src_addr']=='172.16.0.1')
# snort_df_correct.loc[conditional_Wednesday, 'Label'] = 'DoS GoldenEye'

# 周五修正
# 满足条件的Label进行修改
start_time = datetime.strptime('2017-07-07 16:10:00', '%Y-%m-%d %H:%M:%S')
end_time = datetime.strptime('2017-07-07 16:20:00', '%Y-%m-%d %H:%M:%S')
conditional_Friday = (snort_df_filtered['Label'] == 'BENIGN') & (snort_df_filtered['real_time'] >= start_time) & (snort_df_filtered['real_time'] <= end_time) & (snort_df_filtered['dst_addr']=='192.168.10.50') & (snort_df_filtered['dst_port']==80) & (snort_df_filtered['src_addr']=='172.16.0.1')
snort_df_correct.loc[conditional_Friday, 'Label'] = 'DDoS'

start_time = datetime.strptime('2017-07-07 14:50:00', '%Y-%m-%d %H:%M:%S')
end_time = datetime.strptime('2017-07-07 15:30:00', '%Y-%m-%d %H:%M:%S')
conditional_Friday = (snort_df_filtered['Label'] == 'BENIGN') & (snort_df_filtered['real_time'] >= start_time) & (snort_df_filtered['real_time'] <= end_time) & (snort_df_filtered['dst_addr']=='192.168.10.50') & (snort_df_filtered['src_addr']=='172.16.0.1')
snort_df_correct.loc[conditional_Friday, 'Label'] = 'PortScan'

# 修正前的标签结果
print("修正前的标签结果：")
print(snort_df_filtered['Label'].value_counts())
print("修正后的标签结果：")
print(snort_df_correct['Label'].value_counts())

snort_df_correct.to_csv(output_file_path, index=False)

周五，friday-enhance-portscan-alert-label标签结果：
修正前的标签结果：
DDoS        112348
BENIGN       67470
Bot            566
PortScan       429
Name: Label, dtype: int64
修正后的标签结果：
DDoS        115562
BENIGN       64158
Bot            566
PortScan       527
Name: Label, dtype: int64


In [92]:
start_time = datetime.strptime('2017-07-07 14:50:00', '%Y-%m-%d %H:%M:%S')
end_time = datetime.strptime('2017-07-07 15:30:00', '%Y-%m-%d %H:%M:%S')

review_df = snort_df_correct[(snort_df_correct['real_time'] >= start_time) & (snort_df_correct['real_time'] <= end_time)]
# 筛选review_df中key=172.16.0.1_192.168.10.50_54740_80_TCP的行
review_df_2 = review_df.loc[(review_df['dst_addr']=='192.168.10.50') & (review_df['dst_port']==80) & (review_df['src_addr']=='172.16.0.1') & (review_df['Label']=='BENIGN')]
review_df_2

,seconds,action,class,b64_data,dir,dst_ap,eth_dst,eth_len,eth_src,eth_type,...,tos,ttl,udp_len,icmp_code,icmp_id,icmp_seq,icmp_type,real_time,key,Label


In [ ]:
snort_df_filtered.loc[snort_df_filtered['Label']!='BENIGN']